# 04 - Models: Random Forest con TF-IDF + SVD

Este notebook entrena un primer modelo de la familia `04_models_<miembro>` usando **Random Forest** sobre la codificacion **TF-IDF** generada en `02_encoding.ipynb`.

Como los arboles no escalan bien sobre matrices TF-IDF de muy alta dimension, se usa **TruncatedSVD** para reducir dimensionalidad antes del entrenamiento. El flujo mantiene el mismo patron de evaluacion y registro en **MLflow** del notebook `03_baseline.ipynb`.

## 1. Instalacion e imports

In [ ]:
!pip install -q scikit-learn scipy joblib matplotlib seaborn mlflow

In [ ]:
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import load_npz, vstack
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

SEED = 42
MEMBER = "asantiagodiaz"
MLFLOW_URI = "http://ec2-52-5-36-177.compute-1.amazonaws.com:5000"

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")
np.random.seed(SEED)

## 2. Carga de artefactos

Se reutilizan las matrices sparse TF-IDF y labels generadas en `02_encoding.ipynb`.

In [ ]:
X_tr = load_npz('X_tr.npz')
X_val = load_npz('X_val.npz')
X_train_tfidf = load_npz('X_train_tfidf.npz')
X_test_tfidf = load_npz('X_test_tfidf.npz')

y_tr = joblib.load('y_tr.pkl')
y_val = joblib.load('y_val.pkl')
y_train = joblib.load('y_train.pkl')
y_test = joblib.load('y_test.pkl')

print('Shapes cargados:')
print('  X_tr          :', X_tr.shape)
print('  X_val         :', X_val.shape)
print('  X_train_tfidf :', X_train_tfidf.shape)
print('  X_test_tfidf  :', X_test_tfidf.shape)
print('  y_tr          :', len(y_tr))
print('  y_val         :', len(y_val))
print('  y_train       :', len(y_train))
print('  y_test        :', len(y_test))

## 3. Muestreo para entrenamiento del Random Forest

Un `RandomForestClassifier` sobre los 100000 atributos TF-IDF y 1.36M registros completos es demasiado costoso. Para que el experimento sea reproducible y ejecutable, se usa una **muestra estratificada** del train/validation, y se reduce dimensionalidad con SVD.

In [ ]:
TRAIN_SAMPLE_SIZE = 120_000
VAL_SAMPLE_SIZE = 30_000

def stratified_sparse_sample(X, y, sample_size, seed=SEED):
    if sample_size >= X.shape[0]:
        return X, np.asarray(y)

    indices = np.arange(X.shape[0])
    sampled_idx, _ = train_test_split(
        indices,
        train_size=sample_size,
        random_state=seed,
        stratify=y,
    )
    sampled_idx = np.sort(sampled_idx)
    return X[sampled_idx], np.asarray(y)[sampled_idx]

X_tr_sample, y_tr_sample = stratified_sparse_sample(X_tr, y_tr, TRAIN_SAMPLE_SIZE)
X_val_sample, y_val_sample = stratified_sparse_sample(X_val, y_val, VAL_SAMPLE_SIZE)

print('Muestras usadas para tuning:')
print('  X_tr_sample  :', X_tr_sample.shape)
print('  X_val_sample :', X_val_sample.shape)
print('  balance y_tr_sample :', np.bincount(y_tr_sample))
print('  balance y_val_sample:', np.bincount(y_val_sample))

## 4. Busqueda de hiperparametros

Se entrena un pipeline `TruncatedSVD -> RandomForestClassifier` y se selecciona la mejor configuracion por `f1_macro` en validacion.

In [ ]:
param_grid = [
    {'n_components': 200, 'n_estimators': 150, 'max_depth': 20, 'min_samples_leaf': 2},
    {'n_components': 300, 'n_estimators': 150, 'max_depth': 20, 'min_samples_leaf': 2},
    {'n_components': 300, 'n_estimators': 250, 'max_depth': 30, 'min_samples_leaf': 2},
    {'n_components': 400, 'n_estimators': 200, 'max_depth': 30, 'min_samples_leaf': 1},
]

results_rf = []
best_model = None
best_params = None
best_f1 = -1

for params in param_grid:
    pipeline = Pipeline([
        ('svd', TruncatedSVD(n_components=params['n_components'], random_state=SEED)),
        ('rf', RandomForestClassifier(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            min_samples_leaf=params['min_samples_leaf'],
            n_jobs=-1,
            random_state=SEED,
            class_weight='balanced_subsample',
        ))
    ])

    pipeline.fit(X_tr_sample, y_tr_sample)
    y_val_pred = pipeline.predict(X_val_sample)
    y_val_proba = pipeline.predict_proba(X_val_sample)[:, 1]

    metrics_row = {
        **params,
        'accuracy': round(accuracy_score(y_val_sample, y_val_pred), 4),
        'f1_macro': round(f1_score(y_val_sample, y_val_pred, average='macro'), 4),
        'precision_macro': round(precision_score(y_val_sample, y_val_pred, average='macro'), 4),
        'recall_macro': round(recall_score(y_val_sample, y_val_pred, average='macro'), 4),
        'roc_auc': round(roc_auc_score(y_val_sample, y_val_proba), 4),
    }
    results_rf.append(metrics_row)

    if metrics_row['f1_macro'] > best_f1:
        best_f1 = metrics_row['f1_macro']
        best_params = params
        best_model = pipeline

df_rf = pd.DataFrame(results_rf).sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(df_rf)
print('Mejores hiperparametros:', best_params)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = df_rf.copy()
plot_df['config'] = [f"SVD={r.n_components} | Trees={r.n_estimators}" for r in plot_df.itertuples()]
ax.bar(plot_df['config'], plot_df['f1_macro'], color='steelblue')
ax.set_title('Random Forest - F1 macro en validacion')
ax.set_xlabel('Configuracion')
ax.set_ylabel('F1 macro')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 5. Entrenamiento final

Se reentrena el mejor pipeline usando la union de la muestra de train y validacion.

In [ ]:
X_train_final = vstack([X_tr_sample, X_val_sample])
y_train_final = np.concatenate([y_tr_sample, y_val_sample])

model_final = Pipeline([
    ('svd', TruncatedSVD(n_components=best_params['n_components'], random_state=SEED)),
    ('rf', RandomForestClassifier(
        n_estimators=best_params['n_estimators'],
        max_depth=best_params['max_depth'],
        min_samples_leaf=best_params['min_samples_leaf'],
        n_jobs=-1,
        random_state=SEED,
        class_weight='balanced_subsample',
    ))
])

model_final.fit(X_train_final, y_train_final)

print('Modelo final entrenado.')
print('  Train final shape:', X_train_final.shape)
print('  n_components     :', best_params['n_components'])
print('  n_estimators     :', best_params['n_estimators'])
print('  max_depth        :', best_params['max_depth'])

## 6. Evaluacion final en test

In [ ]:
y_pred = model_final.predict(X_test_tfidf)
y_proba = model_final.predict_proba(X_test_tfidf)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')
auc = roc_auc_score(y_test, y_proba)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')

print('Metricas en test:')
print(f'  Accuracy   : {acc:.4f}')
print(f'  F1 macro   : {f1:.4f}')
print(f'  Precision  : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  ROC AUC    : {auc:.4f}')
print()
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negativo (0)', 'Positivo (1)'])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Random Forest + TF-IDF/SVD')
plt.tight_layout()
plt.show()

## 7. Guardar modelo y resultados

In [ ]:
joblib.dump(model_final, 'rf_tfidf_svd_model.pkl')

results_final = pd.DataFrame([{
    'modelo': 'RandomForestClassifier',
    'encoding': 'TF-IDF + TruncatedSVD',
    'member': MEMBER,
    'train_sample_size': len(y_tr_sample),
    'val_sample_size': len(y_val_sample),
    'final_train_size': len(y_train_final),
    'n_components': best_params['n_components'],
    'n_estimators': best_params['n_estimators'],
    'max_depth': best_params['max_depth'],
    'min_samples_leaf': best_params['min_samples_leaf'],
    'accuracy': round(acc, 4),
    'f1_macro': round(f1, 4),
    'roc_auc': round(auc, 4)
}])

results_final.to_csv('results_rf_tfidf_svd.csv', index=False)

print('Archivos guardados:')
print('  rf_tfidf_svd_model.pkl -> pipeline SVD + Random Forest')
print('  results_rf_tfidf_svd.csv -> tabla de metricas')
print()
print(results_final.to_string(index=False))

## 8. Registro en MLflow

In [ ]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('Parcial_1_NLP')

with mlflow.start_run(run_name=f'RandomForest_TFIDF-SVD_{MEMBER}') as run:
    mlflow.set_tags({
        'user': 'Andres Santiago Diaz',
        'member': MEMBER,
        'model_type': 'RandomForestClassifier',
        'encoding': 'TF-IDF + TruncatedSVD',
        'dataset': 'Sentiment140Twitter',
    })

    mlflow.log_params({
        'prep_remove_urls': True,
        'prep_remove_mentions': True,
        'prep_remove_hashtags': True,
        'prep_remove_emojis': True,
        'prep_remove_stopwords': False,
        'prep_lemmatization': False,
    })

    mlflow.log_params({
        'vec_type': 'TfidfVectorizer',
        'vec_max_features': 100000,
        'vec_min_df': 5,
        'vec_max_df': 0.95,
        'vec_ngram_range': '(1,2)',
        'vec_sublinear_tf': True,
    })

    mlflow.log_params({
        'model': 'RandomForestClassifier',
        'dim_reduction': 'TruncatedSVD',
        'n_components': best_params['n_components'],
        'n_estimators': best_params['n_estimators'],
        'max_depth': best_params['max_depth'],
        'min_samples_leaf': best_params['min_samples_leaf'],
        'seed': SEED,
        'train_sample_size': len(y_tr_sample),
        'val_sample_size': len(y_val_sample),
        'final_train_size': len(y_train_final),
        'test_size': X_test_tfidf.shape[0],
        'vocab_size': X_train_tfidf.shape[1],
    })

    y_pred_train = model_final.predict(X_train_final)
    y_proba_train = model_final.predict_proba(X_train_final)[:, 1]
    mlflow.log_metrics({
        'train_accuracy': round(accuracy_score(y_train_final, y_pred_train), 4),
        'train_f1_macro': round(f1_score(y_train_final, y_pred_train, average='macro'), 4),
        'train_precision': round(precision_score(y_train_final, y_pred_train, average='macro'), 4),
        'train_recall': round(recall_score(y_train_final, y_pred_train, average='macro'), 4),
        'train_roc_auc': round(roc_auc_score(y_train_final, y_proba_train), 4),
    })

    y_pred_val_f = best_model.predict(X_val_sample)
    y_proba_val_f = best_model.predict_proba(X_val_sample)[:, 1]
    mlflow.log_metrics({
        'val_accuracy': round(accuracy_score(y_val_sample, y_pred_val_f), 4),
        'val_f1_macro': round(f1_score(y_val_sample, y_pred_val_f, average='macro'), 4),
        'val_precision': round(precision_score(y_val_sample, y_pred_val_f, average='macro'), 4),
        'val_recall': round(recall_score(y_val_sample, y_pred_val_f, average='macro'), 4),
        'val_roc_auc': round(roc_auc_score(y_val_sample, y_proba_val_f), 4),
    })

    mlflow.log_metrics({
        'test_accuracy': round(acc, 4),
        'test_f1_macro': round(f1, 4),
        'test_precision': round(precision, 4),
        'test_recall': round(recall, 4),
        'test_roc_auc': round(auc, 4),
    })

    mlflow.log_artifact('results_rf_tfidf_svd.csv')
    mlflow.sklearn.log_model(
        model_final,
        artifact_path='model',
        registered_model_name=f'RF_TFIDF_SVD_{MEMBER}'
    )

    print('=' * 55)
    print('  Run registrado en MLflow')
    print(f'  Servidor    : {MLFLOW_URI}')
    print('  Experimento : Parcial_1_NLP')
    print(f'  Corrida     : RandomForest_TFIDF-SVD_{MEMBER}')
    print(f'  Run ID      : {run.info.run_id}')
    print(f'  Test F1     : {f1:.4f}')
    print(f'  Test AUC    : {auc:.4f}')
    print('=' * 55)